In [ ]:
# bootstrap: Colab clone + local import of `moelab` (auto-inserted)
import sys, pathlib
if "google.colab" in sys.modules:
    import os, subprocess
    _slug = "aniryou/full-stack-agentic-engineer"
    _repo = pathlib.Path("/content/full-stack-agentic-engineer")
    if not _repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_repo)], check=True)
    os.chdir(_repo / "00-foundations/mixture-of-experts/moe-lab")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
_r = pathlib.Path.cwd().resolve()
while _r != _r.parent and not (_r / "moelab").exists():
    _r = _r.parent
if str(_r) not in sys.path:
    sys.path.insert(0, str(_r))
del _r

# 02 · Watch the router: per-token expert choices, hot experts, and what they do to EP ranks

**Tier:** T1 — a real MoE on one GPU (a free Colab/Kaggle T4 runs OLMoE-1B-7B in fp16 through
transformers, or granite-3.0 MoE; any GPU with `vllm serve --enable-return-routed-experts`).
T0 — bundled traces in vLLM's documented format (**illustrative**: generated by
`tools/make_fixtures.py`, not captured from OLMoE), plus real traces from notebook 01's tiny MoE
when torch is installed. The notebook picks the best source it can find and labels it.

## The one-minute version

* Router decisions are cheap to capture: **forward hooks** on the router modules of a Hugging Face
  MoE (transformers v5 fuses the experts into 3D tensors, so hook the router, not the experts),
  or vLLM's `--enable-return-routed-experts`, which returns a base64 `.npy` of shape
  `(tokens − 1, layers, k)` with every response.
* **Utilisation is uneven**: per layer, a few experts take a multiple of their fair share. vLLM's
  EPLB measures it as *balancedness* = mean load / max load. Part of what you see in a small sample
  is noise — compare against what uniform routing would show at the same sample size.
* **The hot set moves with the traffic**: code, math, prose and other languages light up different
  experts, more so in some layers than others.
* Under expert parallelism each GPU owns a contiguous block of experts, so hot experts that share a
  GPU make that GPU the slowest — and the more GPUs, the less the imbalance averages out.

Concepts: PRIMER §3 "Routing and load balance" (balance at inference: hot experts and domain skew)
and §6 "Running MoE on GPUs" (EP, EPLB) ([`PRIMER.md`](../../PRIMER.md)).

In [ ]:
import os
import numpy as np
from moelab import env, hooks

print(env.describe())
SOURCE = None
if env.server_url():                                          # T1: a vllm server with routed experts
    url, hdr = env.server_url(), env.auth_headers()
    mid = env.served_model(url, hdr)
    E, K = int(os.environ.get("MOELAB_EXPERTS", 64)), int(os.environ.get("MOELAB_TOPK", 8))
    ts = hooks.capture_vllm(url, mid, E, K, headers=hdr)
    SOURCE = f"MEASURED: vLLM at {url} ({mid})"
elif os.environ.get("MOELAB_HF_MODEL") and env.has_gpu() and env.has_transformers():   # T1: HF + hooks
    ts = hooks.capture_hf(os.environ["MOELAB_HF_MODEL"])
    SOURCE = f"MEASURED: transformers + forward hooks ({ts.model})"
else:                                                         # T0
    ts = hooks.load_fixture()
    SOURCE = "ILLUSTRATIVE: " + ts.label
print(SOURCE)
print(f"{ts.model}: {ts.n_experts} experts, top-{ts.top_k}; {len(ts.traces)} sequences, domains {ts.domains()}")

## Worked example: one response, as vLLM returns it

Start the server with `--enable-return-routed-experts`; every chat or completion choice then
carries `routed_experts`. With `"routed_experts_prompt_start": 0` in the request it covers the
prompt too. The last sampled token has not been through the model yet, so there is one row fewer
than tokens.

In [ ]:
if SOURCE.startswith("ILLUSTRATIVE"):
    import json
    doc = json.loads((hooks.FIXTURES / "router_traces_olmoe.json").read_text())
    first = doc["responses"][0]["response"]
    B64 = first["choices"][0]["routed_experts"]
    usage = first["usage"]
    print("server:", doc["server"])
    print("choice keys:", sorted(first["choices"][0]), "| usage:", usage)
    print("routed_experts (base64 .npy):", B64[:60], "...", f"({len(B64):,} chars)")
else:
    B64 = hooks.encode_routed_experts(ts.traces[0].ids)
    usage = {"total_tokens": ts.traces[0].tokens + 1}

## Exercise 2.1 — decode `routed_experts`

Write `decode(b64)`: base64 → bytes → `np.load` of the `.npy` inside (vLLM encodes with
`np.save(..., allow_pickle=False)`; decode the same way, never with pickle).

In [ ]:
import base64
import io


def decode(b64):
    return np.load(io.BytesIO(base64.b64decode(b64)), allow_pickle=False)

In [ ]:
ids0 = decode(B64)
assert ids0.shape == (usage["total_tokens"] - 1, ts.traces[0].layers, ts.top_k), ids0.shape
assert (ids0 == ts.traces[0].ids).all()
probe = np.arange(24, dtype=np.uint16).reshape(2, 3, 4)
assert (decode(hooks.encode_routed_experts(probe)) == probe).all()
print(f"✅ {ids0.shape} = (tokens - 1, layers, top-k); dtype {ids0.dtype}")

## Exercise 2.2 — utilisation per layer

Write `utilisation(ids, n_experts)` → counts `[layers, n_experts]`: how many of the token-slot
assignments of each layer went to each expert. Every row sums to tokens × k.

In [ ]:
def utilisation(ids, n_experts):
    ids = np.asarray(ids)
    return np.stack([np.bincount(ids[:, l, :].ravel(), minlength=n_experts) for l in range(ids.shape[1])])

In [ ]:
allids = ts.stacked()
U = utilisation(allids, ts.n_experts)
assert U.shape == (allids.shape[1], ts.n_experts) and (U.sum(1) == allids.shape[0] * ts.top_k).all()
assert (U == hooks.utilisation(allids, ts.n_experts)).all()
top = hooks.hot_experts(U, 3)
for l in (0, U.shape[0] // 2, U.shape[0] - 1):
    print(f"   layer {l:2d}: hottest {[(e, f'{s:.1%}') for e, s in top[l]]} (fair share {1 / ts.n_experts:.1%})")
print(f"✅ {allids.shape[0]} tokens x {ts.top_k} slots per layer; the hottest expert of a layer takes "
      f"{U.max(1).mean() / U.sum(1).mean():.1%} on average vs a fair {1 / ts.n_experts:.1%}")

## Exercise 2.3 — is that skew real, or a small sample?

vLLM's EPLB logs **balancedness** = mean tokens per expert / max tokens per expert (1.0 perfect).
Even a perfectly uniform router shows balancedness below 1 in a finite sample. Write
`balancedness(counts)` (per row) and `uniform_baseline(tokens, n_experts, k, trials, seed)`: the
mean balancedness of `trials` samples of `tokens` tokens routed uniformly (k *distinct* experts per
token). A layer whose balancedness sits well below the baseline is skewed for real.

In [ ]:
def balancedness(counts):
    c = np.atleast_2d(np.asarray(counts, float))
    return c.mean(axis=1) / c.max(axis=1)


def uniform_baseline(tokens, n_experts, k, trials=50, seed=0):
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(trials):
        picks = np.argsort(rng.random((tokens, n_experts)), axis=1)[:, :k]      # k distinct, uniform
        vals.append(balancedness(np.bincount(picks.ravel(), minlength=n_experts))[0])
    return float(np.mean(vals))

In [ ]:
assert np.allclose(balancedness([[2, 2, 2, 2], [4, 2, 1, 1]]), [1.0, 0.5])
assert np.allclose(balancedness(U), hooks.balancedness(U))
base = uniform_baseline(allids.shape[0], ts.n_experts, ts.top_k)
assert 0.5 < base < 1.0 and uniform_baseline(20_000, 64, 8, trials=5) > base      # more tokens, closer to 1
bal = balancedness(U)
print(f"   uniform routing at {allids.shape[0]} tokens: balancedness {base:.2f}; this trace: "
      f"{bal.min():.2f}-{bal.max():.2f} across layers")
if SOURCE.startswith("ILLUSTRATIVE"):
    assert bal.max() < base                    # the sample was generated with skew
print("✅ judge skew against the uniform baseline at the same sample size, not against 1.0")

## Exercise 2.4 — does the hot set depend on the domain?

Write `js(p, q)`: the Jensen–Shannon divergence in bits between two count vectors
(normalise each, `m = (p + q)/2`, `JS = ½ KL(p‖m) + ½ KL(q‖m)`; 0 = same distribution, 1 =
disjoint). Then compare the domains layer by layer.

In [ ]:
def js(p, q):
    p, q = np.asarray(p, float) / np.sum(p), np.asarray(q, float) / np.sum(q)
    m = (p + q) / 2

    def kl(a, b):
        nz = a > 0
        return float(np.sum(a[nz] * np.log2(a[nz] / b[nz])))
    return 0.5 * kl(p, m) + 0.5 * kl(q, m)

In [ ]:
assert np.isclose(js([1, 2, 3], [2, 4, 6]), 0.0) and np.isclose(js([1, 0], [0, 1]), 1.0)
doms = ts.domains()
per_dom = {d: utilisation(ts.stacked(d), ts.n_experts) for d in doms}
L = U.shape[0]
div = np.array([np.mean([js(per_dom[a][l], per_dom[b][l]) for i, a in enumerate(doms) for b in doms[i + 1:]])
                for l in range(L)])
assert np.allclose(div, hooks.domain_divergence(ts))
print("   mean JS divergence between domains, by layer:", " ".join(f"{v:.2f}" for v in div))
print(f"✅ first quarter of layers {div[:L // 4].mean():.3f}, last quarter {div[-(L // 4):].mean():.3f} "
      f"({'illustrative sample' if SOURCE.startswith('ILLUSTRATIVE') else 'your model'})")

Read it with the sample size in mind: a JS divergence computed from a few hundred tokens per
domain is biased upward by noise (two samples of the *same* distribution rarely give 0). Real
MoEs differ in how much, and in which layers, routing depends on the domain — this is exactly the
kind of claim to measure on your own traffic before building on it.

## Exercise 2.5 — from hot experts to a slow GPU

With `--enable-expert-parallel` and vLLM's default `--expert-placement-strategy linear`, EP rank r
holds experts `[r·E/ep, (r+1)·E/ep)` (`round_robin`, expert e on rank e mod ep, is only honoured
for models with expert groups, like DeepSeek-V3 — vLLM falls back to linear otherwise). Write
`rank_loads(counts, ep)` → `[layers, ep]` assignments per rank under linear placement, then the
**imbalance** max/mean per layer.

In [ ]:
def rank_loads(counts, ep):
    counts = np.atleast_2d(counts)
    return counts.reshape(counts.shape[0], ep, -1).sum(axis=2)

In [ ]:
assert rank_loads([[1, 2, 3, 4]], 2).tolist() == [[3, 7]]
for ep in (2, 4, 8):
    assert (rank_loads(U, ep) == hooks.rank_loads(U, ep, "linear")).all()
imb = {ep: (lambda r: (r.max(1) / r.mean(1)).mean())(rank_loads(U, ep)) for ep in (2, 4, 8, 16)}
print("   mean over layers of (busiest EP rank / mean rank):", {ep: round(v, 2) for ep, v in imb.items()})
assert imb[16] > imb[2]
print("✅ the same routing costs more as EP grows: fewer experts per GPU average less (EPLB replicates hot experts)")

EPLB (`--enable-eplb`, optionally `--eplb-config '{"num_redundant_experts": 32}'` at large scale)
re-places experts every `step_interval` steps from the load it observed over `window_size` steps
(v0.30.0 defaults 3000 and 1000) and can keep extra copies of hot experts — each copy costs HBM on
its rank (layer 02's and 05's primers price it; PRIMER §6 has the formula).

## Worked example: the adapters behind `RouterRecorder`

Router outputs differ by family. `hooks.indices_from_output` knows the three layouts; the check
below builds stand-in modules with the real class names and output layouts (T0: no downloads), so
the same code works on the real models at T1.

In [ ]:
if env.has_torch():
    import torch
    from torch import nn

    def make(name, layout):
        class R(nn.Module):
            def __init__(self):
                super().__init__()
                self.top_k, self.lin = 2, nn.Linear(8, 6, bias=False)

            def forward(self, x):
                logits = self.lin(x)
                idx = logits.topk(2, -1).indices
                w = logits.softmax(-1).gather(-1, idx)
                if layout == "hf":
                    return logits, w, idx                                   # OLMoE, Mixtral, Qwen*, gpt-oss, DeepSeek
                if layout == "granite":
                    return idx, w, logits                                   # GraniteMoeTopKRouter
                scores = torch.full_like(logits, float("-inf")).scatter(1, idx, logits.gather(1, idx)).sigmoid()
                return scores, logits                                       # Llama4Router
        R.__name__ = name
        return R()

    class Tiny(nn.Module):
        def __init__(self):
            super().__init__()
            self.layers = nn.ModuleList([make("OlmoeTopKRouter", "hf"), make("GraniteMoeTopKRouter", "granite"),
                                         make("Llama4Router", "llama4")])

        def forward(self, x):
            return [r(x) for r in self.layers]

    torch.manual_seed(0)
    model, xin = Tiny(), torch.randn(5, 8)
    rec = hooks.RouterRecorder(model, top_k=2)
    model(xin)
    got = rec.pop()
    want = model.layers[0].lin(xin).topk(2, -1).indices.numpy()
    print("recorded", rec.names, got.shape)
    assert all((np.sort(got[:, l], 1) == np.sort(want if l == 0 else model.layers[l].lin(xin).topk(2, -1).indices.numpy(), 1)).all()
               for l in range(3))
    rec.remove()
    print("the three router layouts give the same [tokens, layers, k] array")
else:
    print("T0 without torch: RouterRecorder needs torch; the analysis above is numpy only")

## On a real GPU (T1)

**transformers + hooks** (a free T4 fits OLMoE-1B-7B in fp16 for short prompts; granite-3.0 MoE
models are smaller). Model ids are (verify) on the Hub:

```bash
pip install "transformers>=5" accelerate
MOELAB_HF_MODEL=allenai/OLMoE-1B-7B-0924-Instruct jupyter lab notebooks/02_watch_the_router.ipynb
```

**vLLM** (see [`deploy/any-gpu/`](../deploy/any-gpu/)):

```bash
vllm serve allenai/OLMoE-1B-7B-0924-Instruct --dtype half --max-model-len 4096 \
    --cpu-offload-gb 6 --cpu-offload-params experts --enable-return-routed-experts   # T4: offload to fit
MOELAB_URL=http://127.0.0.1:8000 MOELAB_EXPERTS=64 MOELAB_TOPK=8 jupyter lab ...
```

Then write your own prompts per domain (`hooks.capture_vllm(..., prompts={...})`) — a few hundred
tokens per domain at least, and more before drawing conclusions (Exercise 2.3's baseline tells you
how many).

## In a design review

**Two minutes:** "We record the router, not guess it: forward hooks on the router modules in
transformers, or `--enable-return-routed-experts` in vLLM, give each token's experts per layer.
Per layer, load is uneven — the hottest expert takes a multiple of its fair share — and we judge
that against what uniform routing would show at our sample size before calling it real. The hot
set shifts with the traffic mix, so the placement that balances code traffic may not balance chat.
For serving that matters through expert parallelism: GPU r holds a block of experts, the busiest
GPU sets the step, and the imbalance grows as experts per GPU shrink. The remedies are measured
placement and replication of hot experts (EPLB) — each replica costs HBM — not topic-based
assignment."

**Drill 1.** *Balancedness is 0.6 on 500 tokens. Do we need EPLB?* — Compute the uniform baseline
at 500 tokens first (Exercise 2.3); if it is also ~0.6 you measured noise. Collect more traffic.

**Drill 2.** *Why hook the router instead of the experts in transformers v5?* — The experts are one
fused 3D tensor per layer (`gate_up_proj [E, 2I, d]`), not per-expert modules; the router's output
already names the experts each token uses.

**Drill 3.** *EP=2 looked balanced; at EP=16 the step got slower than expected. Why?* — With 64
experts, EP=16 leaves 4 experts per GPU; one hot expert now dominates its GPU's load, where at EP=2
it averaged with 31 others.